# Pandas: Groupby

Often we don't want a single summary for the whole dataset — we want **one value per group**: the average temperature in each *month*, the strongest earthquake in each *country*, the total rainfall in each *year*. `groupby` is the tool for exactly this, and it's one of the most useful things pandas does.

The pattern it follows is called **split–apply–combine**: split the rows into groups, apply a calculation to each group, then combine the results back together. We'll build that intuition on two datasets — first grouping earthquakes **by country**, then grouping a weather station's daily record **by time** to build climatologies and anomalies. Along the way we'll meet `groupby`'s close relatives **`resample`** (grouping by time interval) and **`rolling`** (moving-window calculations).

These notes draw on the [pandas groupby documentation](http://pandas.pydata.org/pandas-docs/stable/groupby.html); the split-apply-combine framing comes from a [paper by Hadley Wickham](https://www.jstatsoft.org/article/view/v040i01).

:::{admonition} Working through this notebook
:class: tip
This page is a Jupyter notebook. **Download it** using the ⬇ button in the top-right (or copy-paste the cells into a fresh notebook), open it in your environment (JupyterLab on LEAP or Colab), and step through the cells. When you reach a **Try it** admonition, experiment in your own cells before moving on.
:::

:::{admonition} In-class assignment — 10 points
:class: note
The **"Try it"** exercises in this notebook are part of your in-class assignment for this section. Complete them in your own copy of the notebook, push it to your week folder, and post the notebook link on the matching **Courseworks** assignment. (One 10-point assignment covers all the lecture notebooks in this section.)
:::

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = (12,7)
%matplotlib inline
import pandas as pd

First we read the earthquake data from the previous assignment. Two extra steps set up the rest of the lesson: we derive a `country` column from the free-text `place` field, and we split the catalog by magnitude into the larger events (`df`, magnitude > 4) and the smaller ones (`df_small`, magnitude < 4), so we can compare the two groups later.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/earth-DS-ML/summer_2026/refs/heads/main/lectures_DS/data/usgs_earthquakes_2025.csv', parse_dates=['time'], index_col='id')
df['country'] = df.place.str.split(', ').str[-1]
df_small = df[df.mag<4]
df = df[df.mag>4]
df.head()

Before grouping, let's get oriented — how many large events are in `df`, and what does the distribution of magnitudes look like?

In [ ]:
len(df)

In [ ]:
df.mag.plot.hist()

## An Example

What if we wanted to know which **country** had the most earthquakes? `groupby` does this cleanly — it groups all rows that share a value, then we can apply an aggregation (here, `count`) to summarize each group:

In [ ]:
df.groupby('country').mag.count().nlargest(20).plot(kind='bar', figsize=(12,6))

And the same count for the *smaller* events (`df_small`, magnitude below 4) — notice how the ranking of countries shifts:

In [ ]:
df_small.groupby('country').mag.count().nlargest(20).plot(kind='bar', figsize=(12,6))

## What Happened?

Let's break down what `groupby` actually does. The `.groupby(...)` call doesn't immediately compute anything — it returns a `GroupBy` object that holds the grouping logic. Computation happens when you call an aggregation method on it (`.count()`, `.mean()`, `.aggregate(...)`, etc.). This pattern is sometimes called **split-apply-combine**:

1. **Split** the data into groups based on a column or function.
2. **Apply** a function (aggregate, transform, or filter) to each group independently.
3. **Combine** the results back into a single DataFrame or Series.

We can group by any Series — here, the `country` column itself:

In [ ]:
df.country

In [ ]:
df.groupby(df.country)

There is a shortcut for doing this with dataframes: you just pass the column name:

In [ ]:
df.groupby('country')

### The `GroupBy` object

When we call `groupby`, we get back a `GroupBy` object:

In [ ]:
gb = df.groupby('country')
gb

The length tells us how many groups were found:

In [ ]:
len(gb)

All of the groups are available as a dictionary via the `.groups` attribute:

In [ ]:
groups = gb.groups
len(groups)

In [ ]:
groups.keys()

In [ ]:
list(groups.keys())

### Selecting and iterating over groups

The `.groups` attribute is a regular Python dictionary — its keys are the group labels and its values are the row indices in each group. You can look up a single group by name:

In [ ]:
groups['Afghanistan']

You can also loop over the groups directly. Each step of the loop gives you the group's key and the sub-DataFrame of its rows:

In [ ]:
for key, group in gb:
    display(group.head())
    print(f'The key is "{key}"')
    break

Finally, you can pull out one group as its own DataFrame with `get_group`:

In [ ]:
gb.get_group('Chile').head()

:::{admonition} Try it
:class: tip
Take `df`. Use `groupby` to count the number of earthquakes per country and plot the top 10 as a bar chart (`df.groupby('country').mag.count().nlargest(10).plot.bar()`). Then use `gb.get_group('Chile')` to peek at the rows for a specific country.
:::

## Aggregation

Now that we know how to create a `GroupBy` object, let's learn how to do aggregation on it.

One way is to use the `.aggregate` method, which accepts another function as its argument. The result is automatically combined into a new dataframe with the group key as the index.

In [ ]:
gb.aggregate('max').head()

By default, the operation is applied to every column. That's usually not what we want. We can use both `.` or `[]` syntax to select a specific column to operate on. Then we get back a series.

In [ ]:
gb.mag

In [ ]:
gb.mag.aggregate('max')

In [ ]:
gb.mag.aggregate('sum')

In [ ]:
gb.mag.aggregate('mean')

In [ ]:
gb.mag.aggregate('max').nlargest(10)

There are shortcuts for common aggregation functions:

In [ ]:
gb.mag.max().nlargest(10)

In [ ]:
gb.mag.min().nsmallest(10)

In [ ]:
gb.mag.mean().nlargest(10)

In [ ]:
gb.mag.std().nlargest(10)

We can also apply multiple functions at once:

In [ ]:
gb.mag.aggregate(['min', 'max', 'mean']).head()

In [ ]:
gb.mag.aggregate(['min', 'max', 'mean']).nlargest(10, 'mean').plot(kind='bar')

:::{admonition} Try it
:class: tip
Group the earthquakes by country and use `.aggregate(['min', 'max', 'mean'])` on the `mag` column to get all three statistics at once. Then sort by the mean magnitude in descending order and grab the top 10 countries.
:::

## Transformation

The key difference between aggregation and transformation is that aggregation returns a *smaller* object than the original, indexed by the group keys, while *transformation* returns an object with the same index (and same size) as the original object. Groupby + transformation is used when applying an operation that requires information about the whole group.

In this example, we standardize the earthquakes in each country so that the distribution has zero mean and unit variance. We do this by first defining a function called `standardize` and then passing it to the `transform` method.

I admit that I don't know why you would want to do this. `transform` makes more sense to me in the context of time grouping operation. See below for another example.

In [ ]:
def standardize(x):
    return (x - x.mean())/x.std()

mag_standardized_by_country = gb.mag.transform(standardize)
mag_standardized_by_country.head()

:::{admonition} Try it
:class: tip
Define a function `standardize(x)` that returns `(x - x.mean()) / x.std()`. Use it with `df.groupby('country').mag.transform(standardize)` and plot a histogram of the result — magnitudes should now be approximately centered on zero within each country.
:::

## Time Grouping

We already saw how pandas has a strong built-in understanding of time. This capability is even more powerful in the context of `groupby`. With datasets indexed by a pandas `DateTimeIndex`, we can easily group and resample the data using common time units.

To get started, let's load the timeseries data we already explored in past lessons.

In [ ]:
import pooch
import pandas as pd

POOCH = pooch.create(
    path=pooch.os_cache("noaa-data"),
    base_url="doi:10.5281/zenodo.5553029/",
    registry={
        "HEADERS.txt": "md5:2a306ca225fe3ccb72a98953ded2f536",
        "CRND0103-2016-NY_Millbrook_3_W.txt": "md5:eb69811d14d0573ffa69f70dd9c768d9",
        "CRND0103-2017-NY_Millbrook_3_W.txt": "md5:b911da727ba1bdf26a34a775f25d1088",
        "CRND0103-2018-NY_Millbrook_3_W.txt": "md5:5b61bc687261596eba83801d7080dc56",
    },
)

with open(POOCH.fetch("HEADERS.txt")) as fp:
    headers = fp.read().split("\n")[1].split(" ")

dframes = []
for year in range(2016, 2019):
    fname = f"CRND0103-{year}-NY_Millbrook_3_W.txt"
    df = pd.read_csv(POOCH.fetch(fname), parse_dates=[1],
                     names=headers, header=None, sep=r"\s+",
                     na_values=[-9999.0, -99.0])
    dframes.append(df)

df = pd.concat(dframes)
df = df.set_index("LST_DATE")
df.head()

This timeseries has daily resolution, and the daily plots are somewhat noisy.

In [ ]:
df.T_DAILY_MEAN.plot()

A common way to analyze such data in climate science is to create a "climatology," which contains the average values in each month or day of the year. We can do this easily with groupby. Recall that `df.index` is a pandas `DateTimeIndex` object.

In [ ]:
df.index

In [ ]:
df.index.month

In [ ]:
monthly_climatology = df.select_dtypes(include='number').groupby(df.index.month).mean()
monthly_climatology

In [ ]:
monthly_climatology.T_DAILY_MAX.plot()

Each row in this new dataframe represents the average values for the months (1=January, 2=February, etc.)

We can apply more customized aggregations, as with any groupby operation. Below we keep the mean of the mean, max of the max, and min of the min for the temperature measurements.

In [ ]:
monthly_T_climatology = df.groupby(df.index.month).aggregate({'T_DAILY_MEAN': 'mean',
                                                              'T_DAILY_MAX': 'max',
                                                              'T_DAILY_MIN': 'min'})
monthly_T_climatology

In [ ]:
monthly_T_climatology.plot(marker='o')

If we want to do it on a finer scale, we can group by day of year.

In [ ]:
daily_T_climatology = df.groupby(df.index.dayofyear).aggregate({'T_DAILY_MEAN': 'mean',
                                                            'T_DAILY_MAX': 'max',
                                                            'T_DAILY_MIN': 'min'})
daily_T_climatology.plot(marker='.')

### Calculating anomalies

A common mode of analysis in climate science is to remove the climatology from a signal to focus only on the "anomaly" values. This can be accomplished with transformation.

In [ ]:
def remove_climatology(x):
    return x - x.mean()

anomaly = df.groupby(df.index.month).T_DAILY_MEAN.transform(remove_climatology)
anomaly

In [ ]:
anomaly.plot()

:::{admonition} Try it
:class: tip
Compute a **daily** climatology of `T_DAILY_MEAN` by grouping on `df.index.dayofyear` and taking the mean. Plot it as a line chart. Compare its shape to the monthly climatology shown above — the daily curve will be noisier but pick up sub-monthly structure.
:::

### Resampling

We met `resample` at the end of the previous notebook (*Pandas Fundamentals*) as a way to change a time series' resolution. Now that we understand `groupby`, we can name what `resample` actually is: **a `groupby` over time bins**. Instead of grouping rows by the value in a column, it groups them by which time interval they fall into — each month, each year, and so on — then applies an aggregation. It's the same split-apply-combine idea, just with time as the grouping key.

The bin size is given using pandas [offset-alias](http://pandas.pydata.org/pandas-docs/stable/timeseries.html#offset-aliases) syntax (e.g. `'ME'` = month end, `'YE'` = year end). Below we resample to a monthly frequency, taking the mean over each month.

In [ ]:
df.select_dtypes(include='number').resample('ME').mean().plot(y='T_DAILY_MEAN', marker='o')

In [ ]:
df.select_dtypes(include='number').resample('ME').std().plot(y='T_DAILY_MEAN', marker='o')

Just like with `groupby`, we can apply any aggregation function to our `resample` operation.

In [ ]:
df.select_dtypes(include='number').resample('ME').max().plot(y='T_DAILY_MAX', marker='o')

:::{admonition} Try it
:class: tip
Use `df.resample('YE').max()` to get the yearly maximum and plot `T_DAILY_MAX` against the resulting yearly index. Compare with the monthly resampling shown above — what trend, if any, do you see?
:::

### Rolling Operations

The final category of operations applies to "rolling windows". (See [rolling](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.rolling.html) documentation.) We specify a function to apply over a moving window along the index. We specify the size of the window and, optionally, the weights. We also use the keyword `center` to tell pandas whether to center the operation around the midpoint of the window.

In [ ]:
df.rolling(30, center=True).T_DAILY_MEAN.mean().plot()
df.rolling(30, center=True, win_type='triang').T_DAILY_MEAN.mean().plot()

In [ ]:
df.rolling(30, center=True).T_DAILY_MEAN.max().plot()


:::{admonition} Try it
:class: tip
Use `df.rolling(7, center=True).T_DAILY_MEAN.mean()` to compute a 7-day moving average and plot it. Then overlay the raw daily series on the same axes (call `.plot()` twice on the same axes, or use `fig, ax = plt.subplots()` and call `.plot(ax=ax)` for each). What does smoothing reveal that the raw data hides?
:::

## Recap

This notebook was all about **split–apply–combine**:

- **`groupby` on a column** — grouping earthquakes by country, then summarizing each group with **aggregation** (`count`, `mean`, `.aggregate([...])`).
- **Transformation** — returning a value for every row using whole-group information (standardizing within each group; removing a climatology to get anomalies).
- **Time grouping** — grouping by parts of a `DatetimeIndex` to build a **climatology**.
- **`resample`** — a `groupby` over time bins, for changing time resolution.
- **`rolling`** — moving-window calculations for smoothing.

These same `groupby` / `resample` / `rolling` ideas carry straight into the next section, **Xarray**, where they work on labelled N-dimensional arrays.